In [0]:
# Databricks notebook source

# ==========================================
# STEP 1: IMPORT PYSPARK FUNCTIONS
# ==========================================
from pyspark.sql.functions import col, sum, count, countDistinct, avg, round, when

# ==========================================
# STEP 2: DEFINE PATHS & LOAD SILVER
# ==========================================
silver_table_name = "workspace.default.silver_ipl_ball_events"
gold_s3_base_path = "s3://garvit-ipl-data-lake/gold"

silver_df = spark.table(silver_table_name)

# ==========================================
# STEP 3: DATAMART 1 - TEAM PERFORMANCE
# ==========================================
gold_team_performance_df = (
    silver_df
    .groupBy("season", "batting_team")
    .agg(
        sum("runs_total").alias("total_runs"),
        sum("runs_batter").alias("batting_runs"),
        sum("runs_extras").alias("extra_runs"),
        count("*").alias("balls_faced"),
        sum(when(col("is_wicket") == True, 1).otherwise(0)).alias("wickets_lost"),
        sum(when(col("runs_batter") == 4, 1).otherwise(0)).alias("fours"),
        sum(when(col("runs_batter") == 6, 1).otherwise(0)).alias("sixes")
    )
    .withColumnRenamed("batting_team", "team")
    .withColumn("run_rate_per_ball", round(col("total_runs") / col("balls_faced"), 2))
)

(
    gold_team_performance_df.write.format("delta")
                            .mode("overwrite")
                            .option("overwriteSchema", "true")
                            .option("path", f"{gold_s3_base_path}/team_performance")
                            .saveAsTable("workspace.default.gold_team_performance")
)

# ==========================================
# STEP 4: DATAMART 2 - BATTER STATS
# ==========================================
gold_batter_stats_df = (
    silver_df
    .groupBy("season", "batter")
    .agg(
        sum("runs_batter").alias("total_runs"),
        count("*").alias("balls_faced"),
        sum(when(col("runs_batter") == 4, 1).otherwise(0)).alias("fours"),
        sum(when(col("runs_batter") == 6, 1).otherwise(0)).alias("sixes"),
        sum(when(col("is_wicket") == True, 1).otherwise(0)).alias("dismissals")
    )
    .withColumn("strike_rate", round((col("total_runs") / col("balls_faced")) * 100, 2))
    .withColumn(
        "batting_average",
        when(col("dismissals") > 0, round(col("total_runs") / col("dismissals"), 2)).otherwise(None)
    )
)

(
    gold_batter_stats_df.write.format("delta")
                        .mode("overwrite")
                        .option("overwriteSchema", "true")
                        .option("path", f"{gold_s3_base_path}/batter_stats")
                        .saveAsTable("workspace.default.gold_batter_stats")
)

# ==========================================
# STEP 5: DATAMART 3 - BOWLER STATS
# ==========================================
gold_bowler_stats_df = (
    silver_df
    .groupBy("season", "bowler")
    .agg(
        count("*").alias("balls_bowled"),
        sum("runs_total").alias("runs_conceded"),
        sum(when(col("is_wicket") == True, 1).otherwise(0)).alias("wickets")
    )
    .withColumn("overs_bowled", round(col("balls_bowled") / 6, 2))
    .withColumn("economy_rate", round((col("runs_conceded") / col("balls_bowled")) * 6, 2))
)

(
    gold_bowler_stats_df.write.format("delta")
                        .mode("overwrite")
                        .option("overwriteSchema", "true")
                        .option("path", f"{gold_s3_base_path}/bowler_stats")
                        .saveAsTable("workspace.default.gold_bowler_stats")
)

# ==========================================
# STEP 6: DATAMART 4 - VENUE SUMMARY
# ==========================================
gold_venue_summary_df = (
    silver_df
    .groupBy("season", "venue")
    .agg(
        countDistinct("match_id").alias("matches_played"),
        sum("runs_total").alias("total_runs"),
        count("*").alias("total_balls"),
        round(avg("runs_total"), 2).alias("average_runs_per_ball")
    )
    .withColumn("average_runs_per_match", round(col("total_runs") / col("matches_played"), 2))
)

(
    gold_venue_summary_df.write.format("delta")
                         .mode("overwrite")
                         .option("overwriteSchema", "true")
                         .option("path", f"{gold_s3_base_path}/venue_summary")
                         .saveAsTable("workspace.default.gold_venue_summary")
)

# ==========================================
# STEP 7: OPTIMIZE & Z-ORDER GOLD TABLES
# ==========================================
spark.sql("OPTIMIZE workspace.default.gold_batter_stats ZORDER BY (season, batter);")
spark.sql("OPTIMIZE workspace.default.gold_bowler_stats ZORDER BY (season, bowler);")
spark.sql("OPTIMIZE workspace.default.gold_team_performance ZORDER BY (season, team);")

print("✅ Gold analytics datamarts created & Z-Ordered successfully!")

✅ Gold analytics datamarts created & Z-Ordered successfully!
